# Imports


In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification, pipeline, AutoModelForCausalLM
import textstat
import torch
import numpy as np
import re
from lime.lime_text import LimeTextExplainer
import pandas as pd

In [2]:
from ontology_helpers import load_ontology, find_concept, select_ancestors

# Task at hand and integrate ontology

In [3]:
def load_my_classifier():
    print("Stage: LOADING YOUR LOCAL MODEL")

    model_path = "./my_medical_model" 
    
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
    
    print("Stage: LOCAL CLASSIFIER loaded\n")
    return classifier

label_to_class_mapping = {
    'Class_0': 'Neoplasms',
    'Class_1': 'Digestive system diseases',
    'Class_2': 'Nervous system diseases',
    'Class_3': 'Cardiovascular diseases',
    'Class_4': 'General pathological conditions'
}


############################################
# 3. INTEGRATE ONTOLOGY WITH NER OUTPUT & LIME
############################################
def explain_lime_features_with_ontology(text, classifier_pipeline, ontology, user_category, lime_features, ablation_mode):
    # 1. Run Model to get the Class Name (Target)
    results = classifier_pipeline(text[:512], top_k=1) 
    prediction = results[0]
    predicted_id = prediction['label']
    
    # Map "Class_0" -> "Neoplasms"
    predicted_class_name = label_to_class_mapping.get(predicted_id, predicted_id)
    
    print(f"Target Classification: {predicted_class_name} (Confidence: {prediction['score']:.2f})")

    feature_explanations = []

    # 2. Process ONLY Top 6 LIME Features
    if lime_features:
        # Extract top 6 features
        top_features = [f[0] for f in lime_features[:6]]
        print(f"Analyzing LIME Features: {top_features}")

        for feature_word in top_features:
            # Convert numpy string to python string (Fixes the ValueError)
            search_word = str(feature_word) 
            
            # Lookup in Ontology
            feature_concept = find_concept(ontology, search_word)

            if feature_concept:
                # Select ancestors based on User Category (Expert/Beginner)
                feat_ancestors = select_ancestors(
                    entity_concept=feature_concept,
                    user_category=user_category,
                    ablation_mode=ablation_mode
                )
                
                feature_explanations.append({
                    'feature_word': search_word,
                    'ancestors': feat_ancestors
                })
            else:
                continue

    return predicted_class_name, feature_explanations

# Entity Merging

In [4]:
def load_ner_model_for_merging():
    print("Stage: LOADING NER MODEL FOR MERGING")
    tokenizer = AutoTokenizer.from_pretrained("d4data/biomedical-ner-all")
    model = AutoModelForTokenClassification.from_pretrained("d4data/biomedical-ner-all")
    return pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

def merge_entities_in_text(text, ner_pipeline):
    """
    Finds medical entities and replaces spaces with underscores
    so LIME treats them as single tokens.
    """
    entities = ner_pipeline(text)
    
    # Sort by length (descending) to avoid partial replacement issues
    entities = sorted(entities, key=lambda x: len(x['word']), reverse=True)
    
    merged_text = text
    
    for ent in entities:
        original_word = ent['word']
        # Only merge if it's a multi-word entity
        if " " in original_word:
            fused_word = original_word.replace(" ", "_")
            # Safe replacement using regex (case insensitive)
            pattern = re.compile(re.escape(original_word), re.IGNORECASE)
            merged_text = pattern.sub(fused_word, merged_text)
            
    return merged_text

# LLM for Natural Language Explanation

In [5]:
# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading LLM (Qwen2.5-1.5B-Instruct) for explanation generation...")

# Load Tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# Load Model
llm_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM Loaded successfully!")

Loading LLM (Qwen2.5-1.5B-Instruct) for explanation generation...


`torch_dtype` is deprecated! Use `dtype` instead!


LLM Loaded successfully!


In [6]:
SYSTEM_PROMPT = """
You are an explanation generator for a biomedical XAI system.

You do NOT classify text.
You do NOT add external medical knowledge.
You do NOT infer physiology, causation, or mechanisms.

Your ONLY role:
Convert the structured explanation plan (LIME tokens + ontology ancestors +
model prediction) into a clear natural-language explanation.

Rules:
1. Use ONLY the provided ontology triples and model prediction.
2. NEVER add biomedical facts that are not explicitly listed.
3. If a concept has no relation to the predicted class, state this neutrally.
4. Beginner → broad/simple language.
5. Expert → technical terms and ontology hierarchy words.
6. DO NOT write with bullet points. Produce one coherent paragraph.
7. Be concise and strictly grounded.
"""


In [7]:
def generate_classification_prompt(predicted_class, feature_data, user_category):
    """
    feature_data format:
    [
      {
        "feature_word": "ischemia",
        "ancestors": ["vascular disease", "cardiovascular system disease"]
      },
      {
        "feature_word": "tongue",
        "ancestors": ["organ", "sense organ"]
      }
    ]
    """

    # Build structured triple block
    triple_lines = []
    for item in feature_data:
        word = item['feature_word']
        ancestors = item['ancestors']
        
        if len(ancestors) > 0:
            chain = " -> ".join(ancestors)
            triple_lines.append(f"({word} → ancestor → {chain})")
        else:
            triple_lines.append(f"({word} → ancestor → NONE)")
    
    triples_block = "\n".join(triple_lines)

    prompt = f"""
### INPUT
text: [REDACTED FOR BREVITY]
Prediction: {predicted_class}

Salient tokens identified by LIME:
{[item['feature_word'] for item in feature_data]}

Ontology triples (feature → relation → ancestor):
{triples_block}

User type: {user_category}

### OUTPUT REQUIREMENTS
Write a single coherent explanation that:
- States the prediction.
- Uses ONLY the information above.
- Maps each token to its listed ancestors.
- If a token has no biomedical relevance to the prediction, state this without guessing.
- For BEGINNER → use broad/simple language.
- For EXPERT → include hierarchy terms like "anatomical entity", "subclass of", etc.
- Do NOT add medical facts not listed in the triples.
- Do NOT speculate, infer physiology, or make causal claims.

### EXPLANATION:
"""
    return prompt


In [8]:
def generate_classification_reasoning(
    predicted_class,
    feature_data,
    user_category,
    max_new_tokens=180
):
    if not feature_data:
        return (
            f"The model predicted {predicted_class}, "
            "but no ontology-based features were available."
        )

    task_prompt = generate_classification_prompt(
        predicted_class, feature_data, user_category
    )

    # Qwen uses a system + user chat template approach
    full_prompt = llm_tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": task_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(full_prompt, return_tensors="pt").to(llm_model.device)

    output = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,   # LOW TEMP = deterministic, factual
        do_sample=True,
        repetition_penalty=1.1
    )

    decoded = llm_tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract only the model's explanation (after "### EXPLANATION:")
    if "### EXPLANATION:" in decoded:
        explanation = decoded.split("### EXPLANATION:")[-1].strip()
    else:
        explanation = decoded.strip()

    return explanation


----

# Model Loading

In [9]:
ontology = load_ontology("doid.owl")
my_classifier = load_my_classifier()

Stage: LOADING ONTOLOGY from 'doid.owl'
Stage: ONTOLOGY loaded


Device set to use cuda:0


Stage: LOADING YOUR LOCAL MODEL
Stage: LOCAL CLASSIFIER loaded



In [10]:
model_path = "./my_medical_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    top_k=None
)

class_names = [
    "Neoplasms", 
    "Digestive system diseases", 
    "Nervous system diseases", 
    "Cardiovascular diseases", 
    "General pathological conditions"
]

explainer = LimeTextExplainer(class_names=class_names)

Device set to use cuda:0


In [11]:
ner_pipeline = load_ner_model_for_merging()

Stage: LOADING NER MODEL FOR MERGING


Device set to use cuda:0


---

In [12]:
lime_cache = {}

In [13]:
def entity_aware_predictor_batch(texts):
    cleaned = [t.replace("_", " ") for t in texts]

    results = classifier(
        cleaned,
        truncation=True,
        max_length=512
    )

    probs = []
    for result in results:
        sorted_res = sorted(result, key=lambda x: model.config.label2id[x['label']])
        probs.append([x['score'] for x in sorted_res])

    return np.array(probs)


In [14]:
def get_lime_for_text(text, num_features=6, num_samples=300):
    if text in lime_cache:
        return lime_cache[text]

    fused = merge_entities_in_text(text, ner_pipeline)

    exp = explainer.explain_instance(
        fused,
        entity_aware_predictor_batch,
        num_features=num_features,
        num_samples=num_samples
    )

    lime_list = [(w.replace("_", " "), score) for w, score in exp.as_list()]
    lime_cache[text] = lime_list
    return lime_list


# Execution

In [15]:
# user_category = "EXPERT"
# text = "Intravenous nicardipine: an effective new agent for the treatment of severe hypertension. Fifty-six patients with severe hypertension were treated with intravenous nicardipine for infusion periods of eight to twenty-four hours. Each patient achieved satisfactory blood pressure control during the infusion period with a mean controlling dose of 7.85 mg/hr. The dose of nicardipine needed for sustained blood pressure control correlated with untreated diastolic blood pressure but not with untreated systolic blood pressure. These results demonstrate the potential usefulness of intravenous nicardipine for the treatment of severe hypertension requiring rapid lowering, and they suggest also that the severity of pretreatment diastolic hypertension might be a useful indicator of the dose required for blood pressure control."

In [16]:
# lime_features = get_lime_for_text(text)

In [17]:
# predicted_class, feature_data = explain_lime_features_with_ontology(
#     text=text,
#     classifier_pipeline=classifier,
#     ontology=ontology,
#     user_category=user_category,
#     lime_features=lime_features
# )

# for i in feature_data:
#     print(i)

#     # 2. Generate Logic
# explanation = generate_classification_reasoning(
#     predicted_class=predicted_class,
#     feature_data=feature_data,
#     user_category="EXPERT"
# )

# print("\n--- Final Entity-Aware Explanation ---")
# print(explanation)

---

# Result Analysis

In [48]:
user_category = "EXPERT"

text_list = ["Esophageal adenocarcinoma in a patient with surgically treated achalasia. Although squamous cell carcinoma of the esophagus occurs with increased incidence in primary achalasia, esophageal adenocarcinoma has been considered rare in this condition. We report a patient with long-standing achalasia in whom adenocarcinoma of the esophagus occurred many years after Heller esophagomyotomy, presumably related to Barrett's esophagus complicating gastro-esophageal reflux disease.",
            "Multipolar electrocoagulation versus injection therapy in the treatment of bleeding peptic ulcers. A prospective, randomized trial. This study prospectively compares multipolar electrocoagulation and injection therapy in high-risk patients with bleeding ulcers. Patients were considered for entry if they had a bloody nasogastric aspirate, melena, or hematochezia and unstable vital signs, transfusion of greater than or equal to 2 U of blood in 12 hours, or a decrease in hematocrit of greater than or equal to 6% in 12 hours. Sixty patients with endoscopic evidence of an ulcer with active bleeding (n = 26) or a nonbleeding visible vessel (n = 34) were randomly assigned to receive multipolar electrocoagulation or injection with absolute ethanol. Hemostasis was achieved in 14 of 14 actively bleeding patients with multipolar electrocoagulation vs. 10 of 12 (83%) treated with injection. No significant differences were observed between electrocoagulation and injection therapy in any parameter assessed during the hospitalization: incidence of further bleeding (6% vs. 10%), units of blood transfused after treatment (1.8 +/- 0.6 vs. 1.3 +/- 0.4), incidence of surgery for bleeding (6% vs. 7%), length of hospital stay in days (5.8 +/- 0.9 vs. 7.2 +/- 2.5), cost of hospitalization (+7160 +/- +1630 vs. +8520 +/- +2960), or mortality rate (3% vs. 3%). Treatment induced bleeding in nonbleeding visible vessels in 35% of subjects in each group, but this was controlled with continued treatment in all patients. One delayed perforation occurred 9 days after multipolar electrocoagulation. Multipolar electrocoagulation and injection therapy are of comparable efficacy in the treatment of patients with clinical evidence of a major upper gastrointestinal bleed and endoscopic evidence of an ulcer with active bleeding or a nonbleeding visible vessel.",
            "Mycobacterial infection after renal transplantation--report of 14 cases and review of the literature. During a nine-year period, 14 cases of mycobacterial infection (tuberculosis) developed in 403 renal transplant recipients at the King Faisal Specialist Hospital and Research Centre in Riyadh, Saudi Arabia, an incidence of 3.5 per cent. The annual incidence of tuberculosis was about 50 times higher than that in the general population. Infection was disseminated in nine (64.3 per cent), pulmonary in four (28.6 per cent), and genitourinary in 1 (7.1 per cent). In one patient tuberculosis was transmitted by the donor's kidney. The clinical manifestations were often ill-defined and not different from that in the normal host. Cultures from all patients grew Mycobacterium tuberculosis; concomitant infection with other organisms was present in five patients (35.7 per cent). Two of 18 patients (group 1) with positive pretransplant tuberculin skin test developed tuberculosis after transplantation (11 per cent), and neither received isoniazid prophylaxis; three of 70 patients (group 2) with negative skin tests developed tuberculosis after transplantation (4.3 per cent). The difference between the two groups was not statistically significant. Review of all published cases of mycobacterial infections in renal transplant recipients revealed 130 cases. Tuberculosis was disseminated in 38.7 per cent, pulmonary in 40.2 per cent, cutaneous in 12 per cent, and miscellaneous in 9.4 per cent. Atypical mycobacteria were responsible for 29 per cent of disseminated infections, 8 per cent of pulmonary infections and all cases of cutaneous and articular tuberculosis. Invasive procedures were needed to establish the diagnosis in 21 of 33 disseminated cases but in only three of 47 cases of pulmonary tuberculosis (p less than 0.0001). The mortality rate from disseminated disease was 37 per cent and from all other forms of tuberculosis was 11 per cent (p less than 0.005). These findings (1) confirm the higher incidence of tuberculosis in renal transplant recipients, compared to the general population; (2) suggest that pretransplant skin testing probably has little value in identifying patients at risk; (3) show that disseminated tuberculosis is common after renal transplantation and requires invasive procedures for diagnosis; (4) confirm that the donor kidney may be an important source of infection; and (5) indicate that concomitant infection with other organisms is common.",
            "Hyponatraemia secondary to an inappropriately high release of antidiuretic hormone in cardiac tamponade. A spontaneous intrapericardial haemorrhage caused cardiac tamponade in a 29 year old paraplegic man who was being treated with warfarin. The associated persistent hyponatraemia, which was believed to be caused by an inappropriately high release of antidiuretic hormone, rapidly resolved after pericardiocentesis. ",
            "True hermaphrodite with bilateral ovotestes, bilateral gonadoblastomas and dysgerminomas, 46,XX/46,XY karyotype, and a successful pregnancy. The first case (to the authors' knowledge) is reported of a true hermaphrodite with bilateral ovotestes, bilateral gonadoblastomas and dysgerminomas, a 46, XX/46,XY karyotype, and a successful pregnancy. The true hermaphroditism was diagnosed during infancy. The patient was subsequently found to have a gonadoblastoma and a microscopic dysgerminoma in the gonad diagnosed as an ovotestis and excised during infancy. The successful pregnancy occurred when the patient was 29 years old. A year later a large gonadal tumor affecting the remaining gonad was excised. The gonad was found to be an ovotestis, and the tumor was a dysgerminoma arising from a gonadoblastoma. This case further emphasizes the malignant potential of the Y chromosome in patients with abnormal gonads. ",
            "Severe fibromyalgia after hypophysectomy for Cushing's disease. We report a case of fibromyalgia occurring after hypophysectomy for Cushing's disease. Clinical examination revealed tender points at 12 of the 18 tender point sites described in the American College of Rheumatology 1990 criteria for the classification of fibromyalgia. The cause of fibromyalgia remains unknown. In our patient, hypophysectomy may have played a role by disturbing endorphin secretion and pain modulation. ",
            "Urinary tissue factor activity in colorectal disease. Procoagulant activity (PCA) in normal urine has been recognized for over 50 years. Although tissue factor (TF) is produced by certain tumours, and is increased in both tumour-associated macrophages and blood monocytes, the possibility that it might also be increased in urine has not been studied in patients with cancer. We have measured urinary PCA in hospital controls without inflammatory or neoplastic disease (n = 79), in patients with rheumatoid arthritis (n = 8), inflammatory bowel disease (n = 19), colorectal cancer (n = 70) and in patients undergoing colonoscopy (n = 50). Urinary PCA was higher (P less than 0.001) in patients with colorectal cancer and inflammatory bowel disease than controls or patients with rheumatoid arthritis. Fourteen (88 per cent) out of 16 colonoscopy patients subsequently found to have carcinoma or inflammatory bowel disease had levels above the control upper quartile, compared with 8 (24 per cent) out of 34 with normal colonoscopy (P less than 0.001). TF inhibitors confirmed the nature of the PCA and Western blotting studies indicated a urinary TF molecular weight of approximately 38,000. These studies provide further evidence of abnormal haemostasis in malignancy and suggest that determination of urinary TF may provide a useful screening test in patients undergoing colonoscopy. ",
            "Cholangiocarcinoma complicating primary sclerosing cholangitis. Cholangiocarcinoma is more likely to develop in patients with primary sclerosing cholangitis. Our aims were to describe the clinical presentation, course, and management of patients afflicted with both cholangiocarcinoma and primary sclerosing cholangitis and to estimate the prevalence of cholangiocarcinoma in patients with primary sclerosing cholangitis. A retrospective analysis was conducted of 30 patients with both primary sclerosing cholangitis and cholangiocarcinoma managed at our institution during an 8-year period. Development of cholangiocarcinoma was heralded by rapid clinical deterioration with jaundice, weight loss, and abdominal discomfort. Cholangiocarcinoma complicating primary sclerosing cholangitis often was detected at an advanced tumor stage, which precluded effective therapy, and overall median survival was 5 months. Earlier recognition and treatment of cholangiocarcinoma in such patients will be necessary to increase survival rates. Seventy patients with primary sclerosing cholangitis were followed prospectively in a clinical trial of medical therapy for an average of 30 months. Twelve patients died and five were found at autopsy to have cholangiocarcinoma. The potential for cholangiocarcinoma to develop in patients with primary sclerosing cholangitis may indicate that liver transplantation should be considered earlier in the course of the disease. ",
            "An in vitro evaluation of an artificial heart. Interactions between human blood and the Penn State Artificial Heart were examined in vitro to study the effects of various operating conditions on the hematologic response. A dual-loop recirculating flow system that accommodated human blood was developed and blood was subjected for 3 hr to various operating conditions known to alter fluid mechanics in the artificial ventricle. The operating conditions investigated were: 60 beats/min at 50% systolic duration, 60 beats/min at 30% systolic duration, and 90 beats/min at 50% systolic duration. Quantification of plasma free hemoglobin provided a direct indicator of hemolysis in the flow system. Platelet number and beta-thromboglobulin levels were monitored to investigate thrombotic activity, and levels of complement 3a were measured to examine complement system activation. The system was effective in demonstrating the relative hemolytic properties of the operating conditions. Ninety beats/min induced 37% more hemolysis than 60 beats/min at 50% systolic duration, and 50% systolic duration induced 32% more hemolysis than 30% systolic duration at 60 beats/min. There were no statistically significant changes in either platelet number or beta-thromboglobulin levels during the 3 hr recirculation period. Increases were seen in complement 3a levels, but these appeared to be surface-induced and not sensitive to the different operating conditions. These studies demonstrate the usefulness of the flow system in examining the relative hemolytic properties of the artificial ventricle, and suggest that bulk turbulent stresses may play a more important role than laminar wall shear stresses in mediating blood damage in this artificial ventricle. ",
            "Microspectrophotometric DNA analysis in ulcerative colitis with special reference to its application in diagnosis of carcinoma and dysplasia. The deoxyribonucleic (DNA) content was measured by microspectrophotometry in 100 specimens from 60 patients with ulcerative colitis, including six patients in whom the colitis was associated with carcinoma. Some 23 of 30 (77%) specimens of dysplastic tissue showed aneuploidy or polyploidy, whereas 50 of 53 (94%) specimens of non-dysplastic tissue showed diploidy. The difference was statistically significant (p less than 0.001). Polyploidy was often observed in non-dysplastic mucosa from patients who had carcinoma or dysplasia. In the non-dysplastic patients all samples of inflamed tissue showed diploidy. Some 10% of samples without inflammation, however, also showed polyploidy. A good correlation was found between the frequency of polyploid cells and the grade of dysplasia. Microspectrophotometric measurement of DNA content proved useful in the assessment and diagnosis of dysplasia in ulcerative colitis and could be considered for screening high risk patients. ",
            "Reversible cardiogenic shock due to chest tube compression of the right ventricle. A 62-year-old woman developed shock immediately after the insertion of a right-sided chest tube. A chest roentgenogram showed the chest tube to be overlying the heart and possibly compressing the right ventricle. An animal model was developed to replicate this clinical situation. Using a domestic goat model pulmonary artery, peripheral arterial catheters were inserted along with a right sided chest tube placed to suction. A second chest tube guided by a flexible fiberoptic bronchoscope placed within its lumen was positioned between the right ventricle and the sternum of the animals. Thirteen paired measurements in three goats (average of 4.3 measurements per animal) of cardiac output, heart rate, and mean arterial blood pressure were made at baseline and after chest tube placement over the right ventricle. The data were analyzed using a paired t test statistic. Compared with baseline measurements, there was a significant decrease in cardiac output (p less than 0.001) and mean arterial pressure (p less than 0.001) as well as an increase in heart rate (p = 0.0056) after placement of the chest tube across the right ventricle. We conclude that a misplaced chest tube compressing the right ventricle can impede cardiac output and lead to a low cardiac output state. Physicians inserting chest tubes in patients should be aware of this potential complication as it is easily treated by withdrawal of the chest tube. "
            ]

In [49]:
def compute_readability(text):
    return {
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "smog_index": textstat.smog_index(text)
    }

In [50]:
results = []

In [51]:
for text in text_list:
    # print(f"\n--- Processing Text: {text[:50]}... ---")
    
    # 1. Get LIME Features
    lime_features = get_lime_for_text(text)
    for mode in ["full"]:
        # 2. Explain with Ontology
        predicted_class, feature_data = explain_lime_features_with_ontology(
            text=text,
            classifier_pipeline=classifier,
            ontology=ontology,
            user_category=user_category,
            lime_features=lime_features,
            ablation_mode=mode
        )
        print("Feature Data:", feature_data)
        
        # 3. Generate Explanation
        explanation = generate_classification_reasoning(
            predicted_class=predicted_class,
            feature_data=feature_data,
            user_category=user_category
        )
        
        if "assistant\n" in explanation:
            explanation = explanation.split("assistant\n")[-1].strip()

        print("\n--- Final Entity-Aware Explanation ---")
        print(explanation)

        metrics = compute_readability(explanation)

        results.append({
        "text": text[:80] + "...",
        "mode": mode,
        "predicted_class": predicted_class,
        "explanation": explanation,
        **metrics
})

Target Classification: Digestive system diseases (Confidence: 0.57)
Analyzing LIME Features: ['achalasia', 'carcinoma', 'adenocarcinoma', 'reflux', 'disease', 'esophagomyotomy']
Feature Data: [{'feature_word': 'achalasia', 'ancestors': ['disease', 'disease of anatomical entity', 'gastrointestinal system disease', 'esophageal disease', 'achalasia']}, {'feature_word': 'carcinoma', 'ancestors': ['disease', 'disease of cellular proliferation', 'cancer', 'cell type cancer', 'carcinoma']}, {'feature_word': 'adenocarcinoma', 'ancestors': ['disease', 'disease of cellular proliferation', 'cancer', 'cell type cancer', 'carcinoma', 'adenocarcinoma']}, {'feature_word': 'disease', 'ancestors': ['disease']}]

--- Final Entity-Aware Explanation ---
The user's input is redacted for brevity, but it appears to be related to digestive system diseases. The prediction suggests that the user is interested in conditions affecting the digestive tract. Among the salient tokens identified by LIME as relevant to

In [52]:
df = pd.DataFrame(results)
df

,text,mode,predicted_class,explanation,flesch_reading_ease,flesch_kincaid_grade,smog_index
0,Esophageal adenocarcinoma in a patient with su...,full,Digestive system diseases,"The user's input is redacted for brevity, but ...",26.477581,13.966959,16.439396
1,Multipolar electrocoagulation versus injection...,full,Digestive system diseases,The user is experiencing symptoms related to t...,21.977667,15.559926,16.887215
2,Mycobacterial infection after renal transplant...,full,General pathological conditions,"The user's input is redacted for brevity, but ...",3.127426,18.102206,19.537714
3,Hyponatraemia secondary to an inappropriately ...,full,Cardiovascular diseases,"The user's input is redacted for brevity, but ...",4.744333,17.963630,19.287187
4,"True hermaphrodite with bilateral ovotestes, b...",full,Neoplasms,"The prediction is **Neoplasms**, which include...",27.623882,12.946447,14.191786
5,Severe fibromyalgia after hypophysectomy for C...,full,Nervous system diseases,"The user's input is redacted for brevity, but ...",34.788771,12.588861,14.554593
6,Urinary tissue factor activity in colorectal d...,full,Neoplasms,"The user's input is redacted for brevity, but ...",18.703174,16.074618,17.315434
7,Cholangiocarcinoma complicating primary sclero...,full,Digestive system diseases,The prediction is based on the identified toke...,21.912105,14.923158,16.156166
8,An in vitro evaluation of an artificial heart....,full,Cardiovascular diseases,"The user's input contains the term ""hemolysis,...",6.532564,17.730769,18.599290
9,Microspectrophotometric DNA analysis in ulcera...,full,Digestive system diseases,"The user's input is redacted for brevity, but ...",30.400897,12.590163,14.906228


In [53]:
# Save dataframe to CSV
df.to_csv("expert_full_result_analysis.csv", index=False)
print("Saved dataframe to expert_full_result_analysis.csv")

Saved dataframe to expert_full_result_analysis.csv
